# ChillyWatts - Inteligência Energética para Sorveterias
**Hackathon ONE – Projetos G9 | Alura + Oracle**

---

### Sobre o Projeto
O **ChillyWatts** é uma solução focada em **sorveterias** que visa analisar padrões de consumo de energia elétrica, classificar o perfil de eficiência energética e gerar recomendações personalizadas de economia.

### Diferenciais do ChillyWatts
- Foco no segmento de sorveterias (freezers de exibição e armazenamento)
- Modelo treinado com dados sintéticos realistas simulando 1000 estabelecimentos
- Classificação em 3 perfis: **Eficiente**, **Moderado** e **Ineficiente**
- Motor de regras que gera alertas e recomendações contextuais
- Estimativa de custo baseada na tarifa de referência de **R$ 0,75/kWh**

---

### Estrutura do Notebook
1. **EDA** - Exploração e limpeza dos dados
2. **Análise de padrões** - Visualizações dos fatores que impactam o consumo
3. **Transformação de variáveis** - Preparação para os modelos
4. **Treinamento** - Regressão Logística, Árvore de Decisão e Random Forest
5. **Avaliação** - Métricas, relatório de classificação e matriz de confusão
6. **Classificação individual** - Endpoint simplificado (categoria + probabilidade)
7. **Análise completa** - Motor com regras, alertas, custos e recomendações
8. **3 Cenários de teste (RNF04)** - Cenários obrigatórios do edital
9. **Serialização** - Salvamento do modelo e encoders

## 0. Configuração do Ambiente

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
import joblib

%matplotlib inline
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 11

print('Ambiente configurado.')

## 1. Exploração e Limpeza dos Dados (EDA)
Base de dados: `dataset_energia_sorveteria.csv` (1000 amostras sintéticas).

In [ ]:
csv_path = 'dataset_energia_sorveteria.csv'

if not os.path.exists(csv_path):
    import subprocess
    subprocess.run(['python', 'gerador_dados.py'], capture_output=True)

df = pd.read_csv(csv_path)
print(f'Dataset: {df.shape[0]} linhas x {df.shape[1]} colunas')
df.head()

### 1.1 Tipos, Nulos e Duplicatas

In [ ]:
print(df.info())
print('\nNulos:', df.isnull().sum().sum())
print('Duplicatas:', df.duplicated().sum())

### 1.2 Estatísticas Descritivas

In [ ]:
display(df.describe())

print('\nDistribuição das classes de eficiência:')
display(df['categoria_eficiencia'].value_counts())
print(f" Percentuais:\n{df['categoria_eficiencia'].value_counts(normalize=True).mul(100).round(1)}")

## 2. Análise de Padrões de Consumo
Três fatores críticos para sorveterias: sazonalidade, horário de pico e estado das borrachas.

### 2.1 Consumo por Perfil de Eficiência

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='categoria_eficiencia', y='consumo_real_kwh',
            palette='Set2', hue='categoria_eficiencia', legend=False)
plt.title('Consumo Real Mensal (kWh) por Perfil de Eficiência')
plt.tight_layout()
plt.show()

### 2.2 Sazonalidade (Estação do Ano)
No verão, freezers trabalham mais para manter a temperatura ideal, elevando o consumo.

In [ ]:
plt.figure(figsize=(11, 5.5))
sns.barplot(data=df, x='estacao_ano', y='consumo_real_kwh',
            hue='categoria_eficiencia', palette='viridis', errorbar=None)
plt.title('Média de Consumo por Estação do Ano e Perfil de Eficiência')
plt.legend(title='Perfil')
plt.tight_layout()
plt.show()

### 2.3 Uso no Horário de Pico
Abrir freezers no horário de ponta (18h-21h) aumenta o consumo e o custo da energia.

In [ ]:
plt.figure(figsize=(10, 5))
sns.violinplot(data=df, x='uso_horario_pico', y='consumo_real_kwh',
               palette='coolwarm', hue='uso_horario_pico', legend=False)
plt.title('Consumo Real (kWh) vs Intensidade de Uso no Horário de Pico')
plt.tight_layout()
plt.show()

### 2.4 Estado das Borrachas de Vedação
Borrachas gastas fazem o motor do freezer ligar com mais frequência, aumentando o consumo.

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='estado_borracha', y='consumo_real_kwh',
            hue='categoria_eficiencia', palette='Set1')
plt.title('Impacto do Estado da Borracha no Consumo Real')
plt.legend(title='Perfil')
plt.tight_layout()
plt.show()

### 2.5 Correlação entre Variáveis Numéricas

In [ ]:
df_corr = df.copy()
df_corr['estacao_num'] = LabelEncoder().fit_transform(df['estacao_ano'])
df_corr['pico_num'] = LabelEncoder().fit_transform(df['uso_horario_pico'])
df_corr['borracha_num'] = LabelEncoder().fit_transform(df['estado_borracha'])

plt.figure(figsize=(9, 7))
features_corr = ['estacao_num', 'pico_num', 'quantidade_equipamentos',
                 'borracha_num', 'consumo_teorico_estimado_kwh', 'consumo_real_kwh']
sns.heatmap(df_corr[features_corr].corr(), annot=True, cmap='RdBu_r',
            fmt='.2f', linewidths=0.5, vmin=-1, vmax=1)
plt.title('Matriz de Correlação Linear (Pearson)')
plt.tight_layout()
plt.show()

## 3. Tratamento e Transformação de Variáveis
Codificação das variáveis categóricas com `LabelEncoder` e divisão treino/teste (80/20).

In [ ]:
le_estacao = LabelEncoder()
le_pico = LabelEncoder()
le_borracha = LabelEncoder()

df_proc = df.copy()
df_proc['estacao_num'] = le_estacao.fit_transform(df_proc['estacao_ano'])
df_proc['pico_num'] = le_pico.fit_transform(df_proc['uso_horario_pico'])
df_proc['borracha_num'] = le_borracha.fit_transform(df_proc['estado_borracha'])

features_cols = ['estacao_num', 'pico_num', 'quantidade_equipamentos',
                 'borracha_num', 'consumo_teorico_estimado_kwh', 'consumo_real_kwh']
X = df_proc[features_cols]
y = df_proc['categoria_eficiencia']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras')

## 4. Treinamento de Modelos Supervisionados
Três modelos sugeridos pelo edital: Regressão Logística, Árvore de Decisão e Random Forest.

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

dt = DecisionTreeClassifier(max_depth=5, random_state=42)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('Modelos treinados.')

## 5. Avaliação dos Modelos
Comparação de acurácia entre os três modelos.

In [ ]:
print(' Comparativo de Acurácia')
print(f'Regressão Logística: {accuracy_score(y_test, y_pred_lr)*100:.2f}%')
print(f'Árvore de Decisão:   {accuracy_score(y_test, y_pred_dt)*100:.2f}%')
print(f'Random Forest:       {accuracy_score(y_test, y_pred_rf)*100:.2f}%')

### 5.1 Relatório de Classificação (Random Forest)
Métricas detalhadas por classe para o melhor modelo.

In [ ]:
print(classification_report(y_test, y_pred_rf))

### 5.2 Matriz de Confusão

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf, labels=rf.classes_)
fig, ax = plt.subplots(figsize=(7, 5))
ConfusionMatrixDisplay(cm, display_labels=rf.classes_).plot(cmap='Blues', ax=ax)
plt.title('Matriz de Confusão - Random Forest')
plt.grid(False)
plt.show()

## 6. Predição Individual (Formato do Endpoint `/prever`)
Demonstração do formato simplificado de saída, equivalente ao endpoint da API Flask:

**Entrada:** dados do cliente  **→  Saída:** `{categoria, probabilidade}`

In [ ]:
def prever_categoria(dados):
    # Calcula consumo teórico igual ao gerador de dados
    tipo = dados['tipo_predominante']
    tec = dados['tecnologia_predominante']
    if tipo == 'Exibicao' and tec == 'Convencional':
        potencia = 0.25
    elif tipo == 'Exibicao' and tec == 'Inverter':
        potencia = 0.18
    elif tipo == 'Armazenamento' and tec == 'Convencional':
        potencia = 0.15
    else:
        potencia = 0.10
    fator = 1.25 if dados['estado_borracha'] == 'Gasta' else 1.0
    consumo_teorico = round(dados['quantidade_equipamentos'] * (potencia * 24 * 30 * fator), 2)

    input_df = pd.DataFrame([{
        'estacao_num': le_estacao.transform([dados['estacao_ano']])[0],
        'pico_num': le_pico.transform([dados['uso_horario_pico']])[0],
        'quantidade_equipamentos': dados['quantidade_equipamentos'],
        'borracha_num': le_borracha.transform([dados['estado_borracha']])[0],
        'consumo_teorico_estimado_kwh': consumo_teorico,
        'consumo_real_kwh': dados['consumo_real_kwh']
    }])

    categoria = rf.predict(input_df)[0]
    probabilidade = round(float(np.max(rf.predict_proba(input_df))), 2)
    return {'categoria': categoria, 'probabilidade': probabilidade}

# Teste com um exemplo
exemplo = {
    'estabelecimentoId': 'teste_01',
    'estacao_ano': 'Verao',
    'uso_horario_pico': 'Alto',
    'quantidade_equipamentos': 7,
    'tipo_predominante': 'Exibicao',
    'tecnologia_predominante': 'Convencional',
    'estado_borracha': 'Gasta',
    'consumo_real_kwh': 1850.0
}

print('Exemplo de chamada do endpoint /prever:')
print(json.dumps(prever_categoria(exemplo), indent=4, ensure_ascii=False))

## 7. Motor de Análise Completo (Regras + Custos + Recomendações)
Função que integra a predição do ML com regras de negócio para gerar o payload completo de resposta, incluindo alertas, custos e recomendações personalizadas.

In [ ]:
def analisar_cliente_completo(d):
    # --- Consumo Teórico (RN02.2) ---
    tipo, tec = d['tipo_predominante'], d['tecnologia_predominante']
    if tipo == 'Exibicao' and tec == 'Convencional':
        potencia = 0.25
    elif tipo == 'Exibicao' and tec == 'Inverter':
        potencia = 0.18
    elif tipo == 'Armazenamento' and tec == 'Convencional':
        potencia = 0.15
    else:
        potencia = 0.10
    fator = 1.25 if d['estado_borracha'] == 'Gasta' else 1.0
    consumo_teorico = round(d['quantidade_equipamentos'] * (potencia * 24 * 30 * fator), 2)

    # --- Predição ML (RF03 / RN01) ---
    input_df = pd.DataFrame([{
        'estacao_num': le_estacao.transform([d['estacao_ano']])[0],
        'pico_num': le_pico.transform([d['uso_horario_pico']])[0],
        'quantidade_equipamentos': d['quantidade_equipamentos'],
        'borracha_num': le_borracha.transform([d['estado_borracha']])[0],
        'consumo_teorico_estimado_kwh': consumo_teorico,
        'consumo_real_kwh': d['consumo_real_kwh']
    }])
    perfil = rf.predict(input_df)[0]
    prob = round(float(np.max(rf.predict_proba(input_df))), 2)

    # --- Alertas (RN03) ---
    alertas = []
    if d['consumo_real_kwh'] > consumo_teorico * 1.15:
        diff = round(((d['consumo_real_kwh'] - consumo_teorico) / consumo_teorico) * 100)
        alertas.append({'tipo': 'ANOMALIA',
                        'mensagem': f'Consumo real {diff}% acima do esperado para o inventário.'})
    if d['uso_horario_pico'] == 'Alto':
        alertas.append({'tipo': 'HORARIO_PICO',
                        'mensagem': 'Uso crítico de freezers no horário de ponta (18h-21h).'})

    # --- Custos Financeiros (RN04) ---
    tarifa = 0.75
    custo_atual = round(d['consumo_real_kwh'] * tarifa, 2)
    energia_poupada = round(d['consumo_real_kwh'] * 0.20, 2)
    economia_reais = round(energia_poupada * tarifa, 2)

    # --- Recomendações Contextuais ---
    recoms = []
    if d['uso_horario_pico'] in ('Alto', 'Medio'):
        recoms.append('Reduzir a abertura de freezers de exibição entre 18h e 21h, período de tarifa elevada.')
    if d['estacao_ano'] in ('Inverno', 'Outono'):
        recoms.append('No período de menor movimento (inverno/outono), consolidar estoque em menos freezers para reduzir o consumo desnecessário.')
    if d['estado_borracha'] == 'Gasta':
        recoms.append(f'Realizar manutenção preventiva nas borrachas de vedação para evitar perda de eficiência do motor.')
    if d['tecnologia_predominante'] == 'Convencional':
        recoms.append('Avaliar a substituição de equipamentos convencionais por modelos com tecnologia Inverter, que consomem menos energia na partida do motor.')
    if not recoms:
        recoms.append('Perfil eficiente. Continue monitorando os hábitos de consumo periodicamente.')

    return {
        'estabelecimentoId': d['estabelecimentoId'],
        'analise': {
            'perfilEnergetico': perfil,
            'probabilidade': prob,
            'consumoRealKwh': d['consumo_real_kwh'],
            'consumoTeoricoEstimadoKwh': consumo_teorico,
            'custoMensalAtual': custo_atual,
            'energiaEconomizadaPotencialKwh': energia_poupada
        },
        'economiaEstimadaPotencialReais': economia_reais,
        'alertas': alertas,
        'recomendacoes': recoms
    }

## 8. Conjunto Mínimo de Testes (RNF04)
Três cenários obrigatórios para demonstração: Eficiente, Moderado e Ineficiente.

In [ ]:
cenarios = [
    {
        'titulo': 'Cenário 1 - Eficiente',
        'estabelecimentoId': 'loja_eficiente_01',
        'estacao_ano': 'Inverno',
        'uso_horario_pico': 'Baixo',
        'quantidade_equipamentos': 4,
        'tipo_predominante': 'Armazenamento',
        'tecnologia_predominante': 'Inverter',
        'estado_borracha': 'Integra',
        'consumo_real_kwh': 310.0
    },
    {
        'titulo': 'Cenário 2 - Moderado',
        'estabelecimentoId': 'loja_moderado_02',
        'estacao_ano': 'Primavera',
        'uso_horario_pico': 'Medio',
        'quantidade_equipamentos': 5,
        'tipo_predominante': 'Exibicao',
        'tecnologia_predominante': 'Inverter',
        'estado_borracha': 'Integra',
        'consumo_real_kwh': 750.0
    },
    {
        'titulo': 'Cenário 3 - Ineficiente',
        'estabelecimentoId': 'sorveteria_bruno_01',
        'estacao_ano': 'Verao',
        'uso_horario_pico': 'Alto',
        'quantidade_equipamentos': 7,
        'tipo_predominante': 'Exibicao',
        'tecnologia_predominante': 'Convencional',
        'estado_borracha': 'Gasta',
        'consumo_real_kwh': 1850.0
    }
]

for c in cenarios:
    t = c.pop('titulo')
    print(f'\n>>> {t}')
    print(json.dumps(analisar_cliente_completo(c), indent=4, ensure_ascii=False))
    print('-' * 70)

## 9. Serialização do Modelo
Salvando o Random Forest treinado e os LabelEncoders para uso pela API (Flask/Spring Boot).

In [ ]:
joblib.dump(rf, 'modelo_chillywatts.pkl')
joblib.dump(le_estacao, 'le_estacao.pkl')
joblib.dump(le_pico, 'le_pico.pkl')
joblib.dump(le_borracha, 'le_borracha.pkl')

print('Artefatos salvos:')
print(" - 'modelo_chillywatts.pkl' (Random Forest)")
print(" - 'le_estacao.pkl' (LabelEncoder Estação)")
print(" - 'le_pico.pkl' (LabelEncoder Uso no Pico)")
print(" - 'le_borracha.pkl' (LabelEncoder Estado Borracha)")

---
*ChillyWatts - Hackathon ONE G9 | Alura + Oracle*